# Larger systems & higher cardinality

Beyond binary $2\times 2$: ternary / categorical treatments, longer chains,
multi-cause systems, and multi-instrument IV. Uses the literature problem
catalog in `partialid.problems`.


In [1]:
import time
from partialid import Assumptions, Query, Variables, bound, bound_path
from partialid.problems import (
    all_literature_problems,
    chain_order_only,
    shridharan_multi_treatment,
    ternary_cardinalities,
    ternary_exposure,
    two_instruments,
)

def show(label, r, Q=None, dt_ms=None):
    timing = f"  ({dt_ms:.1f} ms)" if dt_ms is not None else ""
    head = Q.describe() if Q is not None else ""
    print(f"{label}{timing}")
    if head:
        print(f"  query: {head}")
    print(f"  [{r.lower:.4f}, {r.upper:.4f}]  w={r.width:.4f}  method={r.method}  tight={r.tight}")


## Ternary confounded exposure

$X \in \{0,1,2\}$, binary $Y$. Under order-only, Manski width $= 1 - P(X=x^\star)$.


In [2]:
p = ternary_exposure(seed=0)
t0 = time.perf_counter()
r = bound(p.variables, p.data, p.query, p.assumptions)
dt = (time.perf_counter() - t0) * 1000
show(p.name, r, p.query, dt)
print(f"  expected_width={p.expected_width:.4f}  |domains|={ {k: len(v) for k,v in p.variables.domains.items()} }")
assert abs(r.width - p.expected_width) < 1e-6


ternary_exposure  (11.6 ms)
  query: P(Y=1 | do(X=1))
  [0.1578, 0.7305]  w=0.5727  method=fiber  tight=True
  expected_width=0.5727  |domains|={'X': 3, 'Y': 2}


## Scaling treatment / outcome cardinality


In [7]:
for card_x, card_y in [(7, 5)]:
    p = ternary_cardinalities(card_x, card_y, seed=1)
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    n_types_proxy = card_x ** 1 * card_y ** card_x  # rough response-type scale for Y|X
    print(
        f"X={card_x} Y={card_y}: w={r.width:.4f}  method={r.method:20s}  "
        f"{dt:6.1f} ms  cells={len(p.data)}"
    )


X=7 Y=5: w=0.8787  method=fiber                 8263.8 ms  cells=35


## MSM sensitivity on a 4-level treatment


In [8]:
p = ternary_cardinalities(4, 2, seed=1)
a = Assumptions.order("X", "Y")
print("Manski:", bound(p.variables, p.data, p.query, a).width)
for lam in (1.0, 1.5, 2.0, 3.0):
    r = bound(p.variables, p.data, p.query, a & Assumptions.tan_msm("X", lam))
    print(f"  Λ={lam:g}: [{r.lower:.4f}, {r.upper:.4f}] w={r.width:.4f}  method={r.method}")


Manski: 0.8585462449087796
  Λ=1: [0.1633, 0.1633] w=0.0000  method=msm_agg/msm_transport_lp
  Λ=1.5: [0.1166, 0.2335] w=0.1169  method=msm_agg/msm_transport_lp
  Λ=2: [0.0932, 0.3036] w=0.2104  method=msm_agg/msm_transport_lp
  Λ=3: [0.0698, 0.4438] w=0.3740  method=msm_agg/msm_transport_lp


## Order-only chains (inert-variable pruning)

Longer chains under `Assumptions.order` stay fast because inert intermediates
are pruned before forming the type LP.


In [9]:
for n, card in [(3, 2), (4, 2), (5, 2), (3, 3), (4, 3)]:
    p = chain_order_only(n, card, seed=7)
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    print(
        f"chain n={n} card={card}: w={r.width:.4f}  method={r.method:12s}  "
        f"{dt:6.1f} ms  vars={list(p.variables.names)}"
    )


chain n=3 card=2: w=0.3742  method=fiber            0.6 ms  vars=['X', 'Z0', 'Y']
chain n=4 card=2: w=0.5198  method=fiber            0.5 ms  vars=['X', 'Z0', 'Z1', 'Y']
chain n=5 card=2: w=0.6574  method=fiber            0.7 ms  vars=['X', 'Z0', 'Z1', 'Z2', 'Y']
chain n=3 card=3: w=0.6267  method=fiber            2.1 ms  vars=['X', 'Z0', 'Y']
chain n=4 card=3: w=0.6270  method=fiber            2.3 ms  vars=['X', 'Z0', 'Z1', 'Y']


## Multi-cause / multi-treatment


In [10]:
for n_t, card in [(2, 2), (3, 2)]:
    p = shridharan_multi_treatment(n_t, card, seed=6)
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    show(p.name, r, p.query, dt)


shridharan_multi_T2_c2  (1.8 ms)
  query: P(Y=1 | do(T0=1))
  [0.3665, 0.7942]  w=0.4277  method=fiber  tight=True
shridharan_multi_T3_c2  (0.7 ms)
  query: P(Y=1 | do(T0=1))
  [0.2130, 0.8945]  w=0.6815  method=fiber  tight=True


## Two-instrument IV


In [11]:
p = two_instruments(seed=3)
t0 = time.perf_counter()
r = bound(p.variables, p.data, p.query, p.assumptions)
dt = (time.perf_counter() - t0) * 1000
show(p.name, r, p.query, dt)
print(f"  |Z1×Z2×X×Y cells|={len(p.data)}")


two_instruments  (19.7 ms)
  query: P(Y=1 | do(X=1))
  [0.3971, 0.7891]  w=0.3920  method=lp  tight=True
  |Z1×Z2×X×Y cells|=16


## Literature catalog sweep

`all_literature_problems()` returns a mid-size battery used in benchmarks.


In [12]:
rows = []
for p in all_literature_problems():
    t0 = time.perf_counter()
    r = bound(p.variables, p.data, p.query, p.assumptions)
    dt = (time.perf_counter() - t0) * 1000
    rows.append((p.name, r.width, r.method, r.tight, dt, len(p.data)))
print(f"{'name':32s} {'width':>8s} {'method':24s} {'tight':5s} {'ms':>8s} {'cells':>5s}")
for name, w, method, tight, dt, cells in rows:
    print(f"{name:32s} {w:8.4f} {method:24s} {str(tight):5s} {dt:8.1f} {cells:5d}")


name                                width method                   tight       ms cells
ternary_exposure                   0.5727 fiber                    True       0.7     6
card_X4_Y2                         0.8585 fiber                    True       1.6     8
card_X3_Y3                         0.6737 fiber                    True       1.6     9
binary_iv                          0.3326 closed/balke_pearl_iv    True       7.8     8
two_instruments                    0.3920 lp                       True      12.9    16
shridharan_multi_T2_c2             0.4277 fiber                    True       1.0     8
chain_n3_c2                        0.3742 fiber                    True       0.5     8
chain_n4_c2                        0.5198 fiber                    True       0.9    16
chain_n3_c3                        0.6267 fiber                    True       1.0    27


## Sensitivity path on ternary exposure


In [13]:
p = ternary_exposure(seed=0)
path = bound_path(
    p.variables, p.data, p.query, Assumptions.manski("X", "Y"),
    param="Lambda", theta_min=1.0, theta_max=4.0, n_grid=7,
)
print(path.summary(at=2.0))
for lam in (1.0, 2.0, 3.0, 4.0):
    lo, hi, w = path.at(lam)
    print(f"  Λ={lam:g}: [{lo:.4f}, {hi:.4f}] w={w:.4f}")


sensitivity path in Lambda
range: [1, 4]
closed_form: False
width @ endpoints: 0 → 0.429513
at Lambda=2: [0.263504, 0.549846] (width 0.286342)
  Λ=1: [0.3692, 0.3692] w=0.0000
  Λ=2: [0.2635, 0.5498] w=0.2863
  Λ=3: [0.2283, 0.6101] w=0.3818
  Λ=4: [0.2106, 0.6402] w=0.4295
